# Oracle PG graph search (manual)

This notebook uses the **same client stack as the backend**: `create_zep_client(...)` with `graph_backend="oracle"`, which builds `GraphitiClient` backed by `graphiti_core.driver.oracle_pg_driver.OraclePGDriver` (see [`client_factory.py`](../app/core/backend_client_factory/client_factory.py)). Search calls mirror [`GraphitiClient.search`](../app/core/backend_client_factory/graphiti/graphiti_client.py) and the `/api/graph/search` route in [`project.py`](../app/core/api/project.py).

**Prerequisites**
- Install backend deps including Graphiti Oracle extras (`graphiti-core[oracle]` per `pyproject.toml`).
- `backend/.env` (or `.env.example` fallback) must set Oracle mode: `ZEP_BACKEND=graphiti`, `GRAPHITI_DB=oracle`, `GRAPHDB_DSN`, `GRAPHDB_USER`, `GRAPHDB_PASSWORD`, embedding/OpenAI vars as needed for hybrid search.
- Set `NOTEBOOK_GRAPH_ID` to the graph id (for Oracle this is the same **project-scoped** id the API uses as `project_id` / `graph_id`).

**Kernel working directory**: the first cell discovers `backend/` by walking up from the notebook cwd (so `backend/`, `backend/notebooks/`, or repo root all work).

In [4]:
import os
import sys
from pathlib import Path

# Walk up from cwd until we find backend root (`app/` + pyproject.toml), or
# `<cwd>/backend` when the kernel starts at the repo root.
_here = Path.cwd().resolve()
BACKEND_ROOT = None
for p in (_here, *_here.parents):
    if (p / "app").is_dir() and (p / "pyproject.toml").is_file():
        BACKEND_ROOT = p
        break
if BACKEND_ROOT is None:
    cand = _here / "backend"
    if (cand / "app").is_dir() and (cand / "pyproject.toml").is_file():
        BACKEND_ROOT = cand
if BACKEND_ROOT is None:
    raise RuntimeError(
        f"Cannot find backend package root (cwd={Path.cwd()}). "
        "Expected an ancestor directory containing app/ and pyproject.toml, or ./backend with those."
    )
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

# Loads backend/.env the same way as the FastAPI app (see app.core.config).
from app.core.config import Config
from app.core.backend_client_factory.client_factory import create_zep_client

print("BACKEND_ROOT", BACKEND_ROOT)
print("GRAPHITI_DB", Config.GRAPHITI_DB)
print("GRAPHDB_DSN set", bool(Config.GRAPHDB_DSN))

BACKEND_ROOT /Users/freedomkwokmacbookpro/Github/imp/zep_graph/backend
GRAPHITI_DB neo4j
GRAPHDB_DSN set False


In [ ]:
# Same resolution idea as project routes: graph id + optional query from env.
GRAPH_ID = (os.environ.get("NOTEBOOK_GRAPH_ID") or os.environ.get("ORACLE_PG_GRAPH_ID") or "1123").strip()
SEARCH_QUERY = (os.environ.get("NOTEBOOK_SEARCH_QUERY") or "test").strip()
LIMIT = int(os.environ.get("NOTEBOOK_SEARCH_LIMIT", "12"))

if not GRAPH_ID:
    raise RuntimeError("Set NOTEBOOK_GRAPH_ID (or ORACLE_PG_GRAPH_ID) to your graph / project id.")

print("GRAPH_ID", GRAPH_ID)
print("SEARCH_QUERY", SEARCH_QUERY)

RuntimeError: Set NOTEBOOK_GRAPH_ID (or ORACLE_PG_GRAPH_ID) to your graph / project id.

In [ ]:
# Mirror create_zep_client path used for Oracle in client_factory (graph_backend forces graphiti + oracle driver).
client = create_zep_client(
    graph_backend="oracle",
    graphiti_embedding_model=Config.GRAPHITI_DEFAULT_EMBEDDING_MODEL,
    project_id=GRAPH_ID,
    enable_otel_tracing=False,
    oracle_pool_min=Config.ORACLE_POOL_MIN,
    oracle_pool_max=Config.ORACLE_POOL_MAX,
    oracle_pool_increment=Config.ORACLE_POOL_INCREMENT,
    oracle_max_coroutines=Config.ORACLE_MAX_COROUTINES,
)
type(client)

In [ ]:
# Node hybrid search (scope="nodes") — same parameter shape as GraphitiClient.search / API client_scope.
node_result = client.search(
    graph_id=GRAPH_ID,
    query=SEARCH_QUERY,
    limit=LIMIT,
    scope="nodes",
    reranker="cross_encoder",
)
print("nodes", len(node_result.nodes))
for n in node_result.nodes[:5]:
    print(getattr(n, "uuid", n), getattr(n, "name", ""), getattr(n, "summary", "")[:80])

In [ ]:
# Edge hybrid search (scope="edges").
edge_result = client.search(
    graph_id=GRAPH_ID,
    query=SEARCH_QUERY,
    limit=LIMIT,
    scope="edges",
    reranker="cross_encoder",
)
print("edges", len(edge_result.edges))
for e in edge_result.edges[:5]:
    print(
        getattr(e, "uuid", e),
        getattr(e, "source_node_uuid", ""),
        "->",
        getattr(e, "target_node_uuid", ""),
        (getattr(e, "fact", "") or "")[:80],
    )

In [ ]:
# Combined search (scope="both") — matches frontend default scope for /api/graph/search.
both_result = client.search(
    graph_id=GRAPH_ID,
    query=SEARCH_QUERY,
    limit=LIMIT,
    scope="both",
    reranker="cross_encoder",
)
print("nodes", len(both_result.nodes), "edges", len(both_result.edges))

In [ ]:
# Release pool / driver if supported.
close = getattr(client, "close", None)
if callable(close):
    close()
    print("client.close() done")
else:
    print("no client.close()")